It's better to test on the converge of the model.

In [167]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [168]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [169]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [170]:
import random
from keras.optimizers import SGD

In [171]:
from sklearn.datasets import make_classification

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

In [172]:
train_pool = 16000
test_size = 500
train_sizes=[1000, 2000, 4000, 8000, 16000]
n_features=160
seed=42

In [173]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           random_state=seed)

In [174]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     -15.782908  -4.132727  12.152943  -5.137928   0.798538 -27.164432   
1       8.337057  -7.322490  -9.760765  -8.599484   9.619708  -4.447431   
2      -2.162488   6.509963   4.350375  -0.122104   6.781976  -7.668239   
3      -2.720892   1.784157   1.809480  -2.197446   0.975266  -5.387154   
4      -1.728961   0.452795  -2.651251  -6.493556   6.239804  -3.990558   
...          ...        ...        ...        ...        ...        ...   
16495  -2.716227   0.058018   2.511306  -5.178930  -6.376654  -5.079567   
16496   0.277247 -11.312607   6.519820   3.821284  -7.619789  -6.264994   
16497  -4.898158  -1.766804 -10.868816  -5.021291   5.341948  11.122732   
16498   0.389821 -13.301590  -1.249690   2.510539   3.759556   1.900669   
16499 -13.536142   1.698689   5.265016  -0.068219   2.659502   6.521925   

       feature_7  feature_8  feature_9  feature_10  ...  feature_153  \
0      12.887783  -8.063796

In [175]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [176]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0 -15.782908  -4.132727  12.152943  -5.137928   0.798538 -27.164432   
1   8.337057  -7.322490  -9.760765  -8.599484   9.619708  -4.447431   
2  -2.162488   6.509963   4.350375  -0.122104   6.781976  -7.668239   
3  -2.720892   1.784157   1.809480  -2.197446   0.975266  -5.387154   
4  -1.728961   0.452795  -2.651251  -6.493556   6.239804  -3.990558   

   feature_7  feature_8  feature_9  feature_10  ...  feature_153  feature_154  \
0  12.887783  -8.063796  17.086774   -5.520991  ...     3.819484   -12.721757   
1   1.474447   5.558629   3.793142    0.989492  ...    -3.817321    -4.204365   
2 -13.248069   2.155376  -0.325390    4.811081  ...     3.133064    12.611964   
3 -12.208869  -1.540317  16.384366    7.957595  ...     2.951633    -1.671340   
4 -12.237140  11.315085   8.821372   -5.751887  ...     0.459658     1.218489   

   feature_155  feature_156  feature_157  feature_158  feature_159  \
0    -4.265044  

In [177]:
features_to_test = 80
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

In [178]:
train_df = nested_train_dfs[4]

In [179]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[-1.5782908e+01 -4.1327267e+00  1.2152943e+01 ... -7.0927677e+00
   4.2685084e+00  1.0000000e-10]
 [ 8.3370571e+00 -7.3224897e+00 -9.7607651e+00 ...  9.5327568e+00
   1.0062770e+01  2.0000000e-10]
 [-2.1624877e+00  6.5099626e+00  4.3503747e+00 ... -6.2190552e+00
   1.6998461e+01  3.0000000e-10]
 ...
 [ 5.1806253e-01 -1.5904241e+01 -2.6893065e+00 ... -6.3524637e+00
  -8.4018952e-01  1.5998000e-06]
 [-2.8096948e+00  9.5520836e-01 -6.7435188e+00 ... -9.5356566e-01
  -1.1646656e+01  1.5999000e-06]
 [ 9.1738653e-01 -3.4890940e+00 -1.6625597e+00 ... -1.1559688e+00
  -3.3634155e+00  1.6000000e-06]]


In [180]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[ 4.1867013e+00 -2.2430241e+00  4.8864728e-01 ... -6.4543724e+00
   4.2893252e+00  1.6000999e-06]
 [ 1.0967977e+01 -1.7221597e+00 -4.9591556e-02 ...  3.8585410e+00
  -9.9176645e+00  1.6002000e-06]
 [-5.9884324e+00 -5.5040576e-02 -4.1153555e+00 ...  6.1255145e+00
  -7.0331001e+00  1.6003000e-06]
 ...
 [-4.8981581e+00 -1.7668041e+00 -1.0868816e+01 ... -1.1036079e+01
  -3.8255043e+00  1.6498000e-06]
 [ 3.8982058e-01 -1.3301590e+01 -1.2496902e+00 ... -5.5559931e+00
   9.6885824e+00  1.6499000e-06]
 [-1.3536141e+01  1.6986887e+00  5.2650166e+00 ... -5.2860934e-01
   2.7125587e+00  1.6499999e-06]]


In [181]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Fold 1

In [182]:
from tensorflow.keras.regularizers import l2

In [183]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 500
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

63/63 - 3s - loss: 0.7749 - accuracy: 0.5529 - val_loss: 0.7448 - val_accuracy: 0.5480 - 3s/epoch - 48ms/step
63/63 - 0s - loss: 0.6998 - accuracy: 0.5841 - val_loss: 0.7067 - val_accuracy: 0.5720 - 405ms/epoch - 6ms/step
63/63 - 0s - loss: 0.6758 - accuracy: 0.5968 - val_loss: 0.6883 - val_accuracy: 0.5940 - 340ms/epoch - 5ms/step
63/63 - 0s - loss: 0.6614 - accuracy: 0.6129 - val_loss: 0.6748 - val_accuracy: 0.6060 - 302ms/epoch - 5ms/step
63/63 - 0s - loss: 0.6502 - accuracy: 0.6241 - val_loss: 0.6634 - val_accuracy: 0.6100 - 283ms/epoch - 4ms/step
63/63 - 0s - loss: 0.6404 - accuracy: 0.6351 - val_loss: 0.6533 - val_accuracy: 0.6180 - 294ms/epoch - 5ms/step
63/63 - 0s - loss: 0.6312 - accuracy: 0.6457 - val_loss: 0.6435 - val_accuracy: 0.6260 - 281ms/epoch - 4ms/step
63/63 - 0s - loss: 0.6222 - accuracy: 0.6557 - val_loss: 0.6341 - val_accuracy: 0.6340 - 292ms/epoch - 5ms/step
63/63 - 0s - loss: 0.6132 - accuracy: 0.6672 - val_loss: 0.6248 - val_accuracy: 0.6500 - 288ms/epoch - 5ms

In [184]:
logits = model.predict(X_test)
per_sample_loss = unreduced_loss_fn(
    tf.convert_to_tensor(y_test), tf.convert_to_tensor(logits)
).numpy()

16/16 [==============================] - 0s 1ms/step


In [185]:
idx_highest = np.argmax(per_sample_loss)

median_loss = np.percentile(per_sample_loss, 50)
idx_median = np.abs(per_sample_loss - median_loss).argmin()

print(idx_highest)
print(idx_median)

310
95


In [186]:
X_high = X_test[idx_highest].reshape(1, -1)
y_high = y_test[idx_highest].reshape(1, -1)

# Median-loss test sample
X_median = X_test[idx_median].reshape(1, -1)
y_median = y_test[idx_median].reshape(1, -1)

print("Highest loss:", per_sample_loss[idx_highest])
print("Median loss:", per_sample_loss[idx_median])

Highest loss: 8.853185
Median loss: 0.0009704886


In [187]:
X_subset = np.vstack([X_high, X_median])
y_subset = np.vstack([y_high, y_median])

# Build a tf.data.Dataset of shape [(x, y), (x, y)]
subset_ds = tf.data.Dataset.from_tensor_slices((X_subset, y_subset))

IF

In [188]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [189]:
num_test_samples = len(subset_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = subset_ds.batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score


df_influence = pd.DataFrame({
    'Train_ID': train_ids,
    'Highest_Loss_Influence': influence_matrix[0],
    'Median_Loss_Influence': influence_matrix[1]
})
print(df_influence)

       Train_ID  Highest_Loss_Influence  Median_Loss_Influence
0             1               -0.059223           5.164135e-05
1             2               -0.006322           5.081561e-06
2             3              -19.342667           1.911187e-02
3             4               -1.277719           4.531833e-04
4             5               -0.252011           1.089729e-04
...         ...                     ...                    ...
15995     15996               -0.001429           1.139963e-06
15996     15997               -0.000009           6.394693e-09
15997     15998               -0.051217           2.103155e-05
15998     15999               -0.951501           3.732230e-04
15999     16000               -1.270225           1.098991e-03

[16000 rows x 3 columns]


TC

In [190]:
num_test_samples = len(subset_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = subset_ds.batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

df_TracIn = pd.DataFrame({
    'Train_ID': train_ids,
    'Highest_Loss_Influence': TracIn_matrix[0],
    'Median_Loss_Influence': TracIn_matrix[1]
})
print(df_TracIn)

       Train_ID  Highest_Loss_Influence  Median_Loss_Influence
0             1            8.420378e-05          -1.679710e-07
1             2            7.145447e-06          -2.138373e-08
2             3            9.527191e-01          -1.345049e-03
3             4            1.376996e-02          -2.821462e-05
4             5            3.520391e-04          -2.685100e-06
...         ...                     ...                    ...
15995     15996            2.462794e-06          -5.510975e-09
15996     15997            1.829733e-08          -4.630753e-11
15997     15998            5.444921e-05          -5.444066e-07
15998     15999           -1.134832e-02           5.371602e-06
15999     16000            2.951323e-03          -7.463688e-06

[16000 rows x 3 columns]


To have more direct view of that, try to match the ranking of two dataframe.

In [191]:
df_TracIn.to_csv("TrainSet_Fragile/TC_Train_Set_Fragile_5.csv",index = False)
df_influence.to_csv("TrainSet_Fragile/IF_Train_Set_Fragile_5.csv",index = False)